# 81城有向链路汇总

- 写入日期：2026-10-17
- 修改日期：2026-10-17
- 本次内容：读取三份年度农机城市接续数据，只保留两端都属于固定81城的记录，按有方向的城市对汇总链路权重。

本 Notebook 属于步骤02，不修改步骤01的三份年度接续数据。输出2021、2022、2023三个年份的主版本和备用版本。

## 1 说明

本步骤使用以下正式口径：

1. 2021、2022、2023年分别计算，不跨年合并。
2. `A→B`和`B→A`是两条不同链路，分别汇总。
3. 主版本的“链路权重”是同一方向城市对全部接续记录的`衰减权重`之和。
4. 备用版本的“链路权重”是同一方向城市对的接续记录数，也就是每条记录计1。
5. 同一台农机多次形成同一方向城市对时，每条记录都参与计算。
6. 当前不按时间间隔、时间冲突、边级或回流标记删除记录。
7. 同城端点继续保留，并使用`是否同城边`和`同城端点记录数`标出。

输出是一行一个“年份—起点城市—终点城市”的已出现有向城市对。没有出现过的城市对不写入文件，其权重可理解为0，但本步骤不主动补零。

## 2 参数

统一设置输入、输出、覆盖保护和辅助函数。Notebook默认拒绝覆盖已有结果；确认重跑时设置环境变量`CITY_LINK_ALLOW_OVERWRITE=1`，代码会先备份旧结果。

In [1]:
from pathlib import Path
from datetime import datetime
import hashlib
import os
import shutil

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_project_root(start_path):
    current_path = Path(start_path).resolve()
    for candidate_path in [current_path, *current_path.parents]:
        if (candidate_path / 'README.md').exists() and (candidate_path / 'Data').exists():
            return candidate_path
    raise FileNotFoundError('没有找到同时包含 README.md 和 Data 文件夹的项目根目录。')


def calculate_sha256(file_path):
    digest = hashlib.sha256()
    with open(file_path, 'rb') as file_handle:
        for file_chunk in iter(lambda: file_handle.read(1024 * 1024), b''):
            digest.update(file_chunk)
    return digest.hexdigest()


def parse_boolean(series, field_name):
    normalized = series.astype('string').str.strip().str.lower()
    invalid_mask = ~normalized.isin(['true', 'false'])
    if invalid_mask.any():
        invalid_values = normalized.loc[invalid_mask].drop_duplicates().tolist()[:10]
        raise ValueError(f'{field_name}发现不能识别的布尔值：{invalid_values}')
    return normalized.eq('true')


def write_csv_atomic(dataframe, target_path):
    temporary_path = target_path.with_name(f'.{target_path.stem}_{run_id}.tmp.csv')
    try:
        dataframe.to_csv(temporary_path, index=False, encoding='utf-8-sig')
        os.replace(temporary_path, target_path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()


project_root = find_project_root(Path.cwd())
analysis_years = [2021, 2022, 2023]
edge_files = {
    year: project_root / f'Data/03_有向边数据/{year}年农机城市接续.csv'
    for year in analysis_years
}
edge_relative_paths = {
    year: f'Data/03_有向边数据/{year}年农机城市接续.csv'
    for year in analysis_years
}
city_list_file = project_root / 'Data/02_筛选后81城数据/citys_2023_1.csv'
manifest_file = project_root / 'Data/data_manifest.csv'
output_data_dir = project_root / 'Data/04_城市链路数据'
check_result_dir = project_root / 'Results/01_链路数据检查'
notebook_relative_path = 'Code/02_构建链路数据/build_city_links.ipynb'

weighted_output_files = {
    year: output_data_dir / f'{year}年衰减链路.csv'
    for year in analysis_years
}
count_output_files = {
    year: output_data_dir / f'{year}年计数链路.csv'
    for year in analysis_years
}
check_output_file = check_result_dir / '链路数据检查.csv'
output_files = [
    *weighted_output_files.values(),
    *count_output_files.values(),
    check_output_file,
]

allow_overwrite = os.getenv('CITY_LINK_ALLOW_OVERWRITE', '0') == '1'
run_time = datetime.now().astimezone()
run_id = run_time.strftime('%Y%m%dT%H%M%S%z')

print(f'项目根目录：{project_root}')
print(f'运行编号：{run_id}')
print(f'允许覆盖已有结果：{allow_overwrite}')

项目根目录：C:\Users\Administrator\Desktop\W
运行编号：20261017T214309+0800
允许覆盖已有结果：True


## 3 读取

读取固定81城名单、数据清单和三份年度城市接续数据。只读取本步骤需要的字段。这里会检查输入文件是否仍与数据清单登记的行数和文件指纹一致。

In [2]:
required_input_columns = {
    '年份', '农机编号', '起点城市', '终点城市', '边级',
    '衰减权重', '是否同城端点', '是否时间冲突'
}
input_column_map = {
    '年份': 'year',
    '农机编号': 'did',
    '起点城市': 'source_city',
    '终点城市': 'target_city',
    '边级': 'edge_level',
    '衰减权重': 'edge_weight',
    '是否同城端点': 'is_same_city_endpoint',
    '是否时间冲突': 'has_time_conflict',
}

for required_file in [city_list_file, manifest_file, *edge_files.values()]:
    if not required_file.exists():
        raise FileNotFoundError(f'缺少必要文件：{required_file}')

city_list = pd.read_csv(city_list_file, encoding='utf-8-sig', dtype='string')
if list(city_list.columns) != ['city']:
    raise ValueError(f'固定81城名单字段应为 city，实际为：{city_list.columns.tolist()}')
city_list['city'] = city_list['city'].str.strip()
if city_list['city'].isna().any() or city_list['city'].eq('').any():
    raise ValueError('固定81城名单存在空城市名称。')
if len(city_list) != 81 or city_list['city'].nunique() != 81:
    raise ValueError(f'固定城市名单应为81行、81个唯一城市，实际为{len(city_list)}行、{city_list["city"].nunique()}个唯一城市。')
city_set = set(city_list['city'])

manifest = pd.read_csv(manifest_file, encoding='utf-8-sig', dtype='string')
manifest_by_path = manifest.set_index('relative_path', drop=False)

input_stats = []
aggregated_internal = {}

for year in analysis_years:
    edge_file = edge_files[year]
    relative_path = edge_relative_paths[year]
    if relative_path not in manifest_by_path.index:
        raise ValueError(f'数据清单没有登记正式输入：{relative_path}')
    manifest_row = manifest_by_path.loc[relative_path]
    if manifest_row['formal_status'] != '正式':
        raise ValueError(f'数据清单未把输入标记为正式：{relative_path}')
    if calculate_sha256(edge_file) != manifest_row['sha256']:
        raise ValueError(f'输入文件内容与数据清单登记不一致：{relative_path}')

    header_columns = pd.read_csv(edge_file, encoding='utf-8-sig', nrows=0).columns.tolist()
    missing_columns = sorted(required_input_columns - set(header_columns))
    if missing_columns:
        raise ValueError(f'{relative_path}缺少必要字段：{missing_columns}')

    records = pd.read_csv(
        edge_file,
        encoding='utf-8-sig',
        usecols=list(required_input_columns),
        dtype={'农机编号': 'string', '起点城市': 'string', '终点城市': 'string'},
    ).rename(columns=input_column_map)

    expected_row_count = int(manifest_row['row_count'])
    if len(records) != expected_row_count:
        raise ValueError(f'{relative_path}行数应为{expected_row_count:,}，实际为{len(records):,}。')

    records['source_city'] = records['source_city'].str.strip()
    records['target_city'] = records['target_city'].str.strip()
    records['year'] = pd.to_numeric(records['year'], errors='coerce').astype('Int64')
    records['edge_level'] = pd.to_numeric(records['edge_level'], errors='coerce').astype('Int64')
    records['edge_weight'] = pd.to_numeric(records['edge_weight'], errors='coerce')
    records['is_same_city_endpoint'] = parse_boolean(records['is_same_city_endpoint'], '是否同城端点')
    records['has_time_conflict'] = parse_boolean(records['has_time_conflict'], '是否时间冲突')

    critical_columns = ['year', 'did', 'source_city', 'target_city', 'edge_level', 'edge_weight']
    missing_critical = records[critical_columns].isna().sum()
    if int(missing_critical.sum()) != 0:
        raise ValueError(f'{relative_path}关键字段存在缺失：{missing_critical[missing_critical > 0].to_dict()}')
    if not records['year'].eq(year).all():
        raise ValueError(f'{relative_path}包含不属于{year}年的记录。')
    if not records['edge_level'].ge(1).all():
        raise ValueError(f'{relative_path}发现边级小于1的记录。')

    expected_weights = np.power(0.5, records['edge_level'].astype(int) - 1)
    if not np.allclose(records['edge_weight'], expected_weights, rtol=0, atol=1e-12):
        raise ValueError(f'{relative_path}存在不符合0.5^(边级-1)的衰减权重。')
    if not records['is_same_city_endpoint'].eq(records['source_city'].eq(records['target_city'])).all():
        raise ValueError(f'{relative_path}的同城端点标记与城市名称不一致。')

    in_81_mask = records['source_city'].isin(city_set) & records['target_city'].isin(city_set)
    in_81_records = records.loc[in_81_mask].copy()
    excluded_record_count = int((~in_81_mask).sum())
    assert len(in_81_records) + excluded_record_count == len(records), f'{year}年筛选前后数量不能还原。'
    in_81_records['is_first_level'] = in_81_records['edge_level'].eq(1).astype(int)
    in_81_records['is_multi_level'] = in_81_records['edge_level'].gt(1).astype(int)
    in_81_records['same_city_record'] = in_81_records['is_same_city_endpoint'].astype(int)
    in_81_records['time_conflict_record'] = in_81_records['has_time_conflict'].astype(int)

    grouped = (
        in_81_records.groupby(['source_city', 'target_city'], sort=True, observed=True)
        .agg(
            decay_weight_sum=('edge_weight', 'sum'),
            record_count=('did', 'size'),
            machine_count=('did', 'nunique'),
            first_level_count=('is_first_level', 'sum'),
            multi_level_count=('is_multi_level', 'sum'),
            min_edge_level=('edge_level', 'min'),
            max_edge_level=('edge_level', 'max'),
            same_city_record_count=('same_city_record', 'sum'),
            time_conflict_count=('time_conflict_record', 'sum'),
        )
        .reset_index()
    )
    grouped['year'] = year
    grouped['directed_edge_id'] = grouped['source_city'] + '→' + grouped['target_city']
    grouped['is_same_city_link'] = grouped['source_city'].eq(grouped['target_city'])
    grouped['source_file'] = relative_path
    aggregated_internal[year] = grouped

    non_self = grouped.loc[~grouped['is_same_city_link'], ['source_city', 'target_city']]
    reversed_pairs = non_self.merge(
        non_self,
        left_on=['source_city', 'target_city'],
        right_on=['target_city', 'source_city'],
        how='inner',
    )
    bidirectional_unordered_pair_count = len(reversed_pairs) // 2

    input_stats.append({
        'year': year,
        'input_row_count': len(records),
        'in_81_record_count': len(in_81_records),
        'excluded_record_count': excluded_record_count,
        'directed_pair_count': len(grouped),
        'machine_count': int(in_81_records['did'].nunique()),
        'first_level_count': int(in_81_records['is_first_level'].sum()),
        'multi_level_count': int(in_81_records['is_multi_level'].sum()),
        'same_city_record_count': int(in_81_records['same_city_record'].sum()),
        'same_city_pair_count': int(grouped['is_same_city_link'].sum()),
        'time_conflict_count': int(in_81_records['time_conflict_record'].sum()),
        'decay_weight_sum': float(in_81_records['edge_weight'].sum()),
        'bidirectional_unordered_pair_count': int(bidirectional_unordered_pair_count),
        'input_sha256': calculate_sha256(edge_file),
    })

input_summary = pd.DataFrame(input_stats)
display(input_summary)

,year,input_row_count,in_81_record_count,excluded_record_count,directed_pair_count,machine_count,first_level_count,multi_level_count,same_city_record_count,same_city_pair_count,time_conflict_count,decay_weight_sum,bidirectional_unordered_pair_count,input_sha256
0,2021,110458,96536,13922,3281,10904,34104,62432,7388,80,0,51440.146837,798,7985254330ed1795fa3d7178d818f8bcfe73f20384b06f...
1,2022,275055,257123,17932,3653,28602,89026,168097,24699,81,5,134057.924479,1065,ba07c3db48ff7f227dceac4381735802b5971ea53c2cac...
2,2023,500521,441378,59143,4348,40354,141726,299652,41220,81,19,219648.344314,1338,a5c29cb0d8b16908934735c686ed98cacefa44ba4bd213...


## 4 处理

根据相同的“年份—起点城市—终点城市”分组生成两个版本。两个版本的城市对、接续记录数和检查字段完全相同，只有`链路权重`与`权重口径`不同。

In [3]:
output_column_map = {
    'year': '年份',
    'directed_edge_id': '有向边编号',
    'source_city': '起点城市',
    'target_city': '终点城市',
    'link_weight': '链路权重',
    'weight_method': '权重口径',
    'record_count': '接续记录数',
    'machine_count': '涉及农机数',
    'first_level_count': '一级边记录数',
    'multi_level_count': '二级及以上边记录数',
    'min_edge_level': '最小边级',
    'max_edge_level': '最大边级',
    'same_city_record_count': '同城端点记录数',
    'time_conflict_count': '时间冲突记录数',
    'is_same_city_link': '是否同城边',
    'source_file': '来源文件',
}
output_column_order = list(output_column_map.values())


def make_output_table(grouped_data, weight_method):
    output_data = grouped_data.copy()
    if weight_method == '衰减权重求和':
        output_data['link_weight'] = output_data['decay_weight_sum']
    elif weight_method == '每条接续记录计1':
        output_data['link_weight'] = output_data['record_count'].astype(int)
    else:
        raise ValueError(f'未知权重口径：{weight_method}')
    output_data['weight_method'] = weight_method
    output_data = output_data.rename(columns=output_column_map)
    return output_data[output_column_order].sort_values(['起点城市', '终点城市']).reset_index(drop=True)


weighted_outputs = {
    year: make_output_table(aggregated_internal[year], '衰减权重求和')
    for year in analysis_years
}
count_outputs = {
    year: make_output_table(aggregated_internal[year], '每条接续记录计1')
    for year in analysis_years
}

output_summary = pd.DataFrame([
    {
        '年份': year,
        '81城内接续记录数': int(input_summary.loc[input_summary['year'].eq(year), 'in_81_record_count'].iloc[0]),
        '有向城市对数': len(weighted_outputs[year]),
        '衰减权重合计': weighted_outputs[year]['链路权重'].sum(),
        '计数权重合计': count_outputs[year]['链路权重'].sum(),
    }
    for year in analysis_years
])
display(output_summary.style.format({'衰减权重合计': '{:,.6f}', '计数权重合计': '{:,.0f}'}).hide(axis='index'))

年份,81城内接续记录数,有向城市对数,衰减权重合计,计数权重合计
2021,96536,3281,"51,440.146837","96,536"
2022,257123,3653,"134,057.924479","257,123"
2023,441378,4348,"219,648.344314","441,378"


## 5 检查

写文件前检查城市范围、方向、主键、两种权重、一级与多级记录拆分、同城端点和时间冲突统计。任一关键检查不通过，程序会停止，不生成正式结果。

In [4]:
for year in analysis_years:
    grouped = aggregated_internal[year]
    weighted_data = weighted_outputs[year]
    count_data = count_outputs[year]
    year_stats = input_summary.loc[input_summary['year'].eq(year)].iloc[0]

    assert not grouped[['source_city', 'target_city']].duplicated().any(), f'{year}年有向城市对重复。'
    assert not weighted_data.isna().any().any(), f'{year}年衰减版本存在缺失值。'
    assert not count_data.isna().any().any(), f'{year}年计数版本存在缺失值。'
    assert grouped['source_city'].isin(city_set).all(), f'{year}年输出含81城外起点。'
    assert grouped['target_city'].isin(city_set).all(), f'{year}年输出含81城外终点。'
    assert (grouped['record_count'] == grouped['first_level_count'] + grouped['multi_level_count']).all(), f'{year}年一级与多级记录数不能还原总数。'
    assert grouped['is_same_city_link'].eq(grouped['source_city'].eq(grouped['target_city'])).all(), f'{year}年同城边标记错误。'
    assert grouped.loc[grouped['is_same_city_link'], 'same_city_record_count'].eq(grouped.loc[grouped['is_same_city_link'], 'record_count']).all(), f'{year}年同城边记录数错误。'
    assert grouped.loc[~grouped['is_same_city_link'], 'same_city_record_count'].eq(0).all(), f'{year}年非同城边出现同城端点记录。'
    assert grouped['time_conflict_count'].le(grouped['record_count']).all(), f'{year}年时间冲突记录数超过总记录数。'
    assert grouped['min_edge_level'].ge(1).all(), f'{year}年最小边级小于1。'
    assert grouped['max_edge_level'].ge(grouped['min_edge_level']).all(), f'{year}年最大边级小于最小边级。'

    assert weighted_data[['起点城市', '终点城市']].equals(count_data[['起点城市', '终点城市']]), f'{year}年两个版本的城市对不一致。'
    assert weighted_data['接续记录数'].equals(count_data['接续记录数']), f'{year}年两个版本的记录数不一致。'
    assert np.allclose(count_data['链路权重'], count_data['接续记录数'], rtol=0, atol=0), f'{year}年计数版本没有做到每条记录计1。'
    assert (weighted_data['链路权重'] > 0).all(), f'{year}年衰减版本出现非正权重。'
    assert (weighted_data['链路权重'] <= weighted_data['接续记录数'] + 1e-12).all(), f'{year}年衰减权重大于记录数。'
    assert np.isclose(weighted_data['链路权重'].sum(), year_stats['decay_weight_sum'], rtol=0, atol=1e-9), f'{year}年衰减权重合计与81城内输入不一致。'
    assert int(count_data['链路权重'].sum()) == int(year_stats['in_81_record_count']), f'{year}年计数权重合计与81城内输入行数不一致。'

print('写出前检查全部通过。')

写出前检查全部通过。


## 6 输出

把两个口径按年份分别写入CSV，并生成一份简单的运行检查记录。CSV统一使用UTF-8-SIG编码。写出后重新读入文件，核对行数、字段、城市对和权重合计。

In [5]:
existing_outputs = [output_path for output_path in output_files if output_path.exists()]
backup_dir = None
if existing_outputs and not allow_overwrite:
    existing_text = '\n'.join(str(path) for path in existing_outputs)
    raise FileExistsError(
        '以下正式结果已经存在，Notebook默认不覆盖：\n'
        f'{existing_text}\n'
        '确认重跑时，请设置环境变量 CITY_LINK_ALLOW_OVERWRITE=1。'
    )

if existing_outputs:
    backup_dir = project_root / 'Archive' / f'{run_time.strftime("%Y-%m-%d_%H%M%S")}_链路数据重跑前'
    for output_path in existing_outputs:
        relative_output_path = output_path.relative_to(project_root)
        backup_path = backup_dir / relative_output_path
        backup_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(output_path, backup_path)

output_data_dir.mkdir(parents=True, exist_ok=True)
check_result_dir.mkdir(parents=True, exist_ok=True)

for year in analysis_years:
    write_csv_atomic(weighted_outputs[year], weighted_output_files[year])
    write_csv_atomic(count_outputs[year], count_output_files[year])

check_rows = []

def add_check(year, metric_group, metric, value, unit, status, note):
    check_rows.append({
        '运行编号': run_id,
        '运行时间': run_time.isoformat(timespec='seconds'),
        '年份': year,
        '指标组': metric_group,
        '指标': metric,
        '数值': value,
        '单位': unit,
        '检查状态': status,
        '说明': note,
    })

for year in analysis_years:
    stats = input_summary.loc[input_summary['year'].eq(year)].iloc[0]
    weighted_readback = pd.read_csv(weighted_output_files[year], encoding='utf-8-sig')
    count_readback = pd.read_csv(count_output_files[year], encoding='utf-8-sig')

    if weighted_readback.columns.tolist() != output_column_order or count_readback.columns.tolist() != output_column_order:
        raise ValueError(f'{year}年输出字段顺序不正确。')
    if len(weighted_readback) != int(stats['directed_pair_count']) or len(count_readback) != int(stats['directed_pair_count']):
        raise ValueError(f'{year}年输出读回行数不正确。')
    if weighted_readback[['起点城市', '终点城市']].duplicated().any() or count_readback[['起点城市', '终点城市']].duplicated().any():
        raise ValueError(f'{year}年输出读回后发现重复城市对。')
    key_columns = ['年份', '起点城市', '终点城市']
    expected_key_values = weighted_outputs[year][key_columns].astype('string').to_numpy()
    weighted_key_values = weighted_readback[key_columns].astype('string').to_numpy()
    count_key_values = count_readback[key_columns].astype('string').to_numpy()
    if not np.array_equal(weighted_key_values, expected_key_values):
        raise ValueError(f'{year}年衰减版本读回后的城市对集合或顺序不一致。')
    if not np.array_equal(count_key_values, expected_key_values):
        raise ValueError(f'{year}年计数版本读回后的城市对集合或顺序不一致。')
    if weighted_readback.isna().any().any() or count_readback.isna().any().any():
        raise ValueError(f'{year}年输出读回后发现缺失值。')
    if not np.isclose(weighted_readback['链路权重'].sum(), stats['decay_weight_sum'], rtol=0, atol=1e-9):
        raise ValueError(f'{year}年衰减版本读回后的权重合计不一致。')
    if int(count_readback['链路权重'].sum()) != int(stats['in_81_record_count']):
        raise ValueError(f'{year}年计数版本读回后的权重合计不一致。')

    add_check(year, '输入', '年度接续记录数', int(stats['input_row_count']), '行', '通过', '与数据清单登记行数一致')
    add_check(year, '筛选', '两端都在81城的记录数', int(stats['in_81_record_count']), '行', '通过', '进入本步骤链路汇总')
    add_check(year, '筛选', '至少一端不在81城的记录数', int(stats['excluded_record_count']), '行', '已记录', '本步骤不进入81城链路汇总')
    add_check(year, '输出', '有向城市对数', int(stats['directed_pair_count']), '对', '通过', '起点和终点顺序不同视为不同城市对')
    add_check(year, '输出', '涉及农机数', int(stats['machine_count']), '台', '通过', '两端都在81城记录中的不同农机数')
    add_check(year, '边级', '一级边记录数', int(stats['first_level_count']), '行', '已记录', '相邻访问形成的记录')
    add_check(year, '边级', '二级及以上边记录数', int(stats['multi_level_count']), '行', '已记录', '中间隔有其他访问位置的记录')
    add_check(year, '保留项', '同城端点记录数', int(stats['same_city_record_count']), '行', '已记录', '当前保留并单独标记')
    add_check(year, '保留项', '同城有向边数', int(stats['same_city_pair_count']), '对', '已记录', '一个城市最多对应一条同城边')
    add_check(year, '保留项', '时间冲突记录数', int(stats['time_conflict_count']), '行', '已记录', '当前不删除')
    add_check(year, '方向', '同时存在两个方向的城市组合数', int(stats['bidirectional_unordered_pair_count']), '组', '通过', 'A到B和B到A分别保存')
    add_check(year, '权重', '衰减权重合计', float(stats['decay_weight_sum']), '权重', '通过', '等于81城内输入记录的衰减权重之和')
    add_check(year, '权重', '计数权重合计', int(stats['in_81_record_count']), '权重', '通过', '每条接续记录计1')
    add_check(year, '文件核对', '输入文件SHA-256', stats['input_sha256'], '文件指纹', '通过', edge_relative_paths[year])

    for version_name, output_path, output_data in [
        ('衰减版本', weighted_output_files[year], weighted_readback),
        ('计数版本', count_output_files[year], count_readback),
    ]:
        add_check(year, '文件核对', f'{version_name}文件大小', output_path.stat().st_size, '字节', '通过', str(output_path.relative_to(project_root)))
        add_check(year, '文件核对', f'{version_name}文件SHA-256', calculate_sha256(output_path), '文件指纹', '通过', '文件内容指纹')
        add_check(year, '文件核对', f'{version_name}输出行数', len(output_data), '行', '通过', '读回行数与有向城市对数一致')

add_check('全部', '代码', '生成Notebook', notebook_relative_path, '路径', '通过', '全部处理代码位于Notebook中')
add_check('全部', '覆盖保护', '旧输出备份目录', str(backup_dir.relative_to(project_root)) if backup_dir else '本次没有旧输出', '路径', '通过', '有旧输出时先备份再覆盖')

check_output = pd.DataFrame(check_rows)
write_csv_atomic(check_output, check_output_file)

if check_output_file.read_bytes()[:3] != b'\xef\xbb\xbf':
    raise ValueError('链路数据检查.csv不是UTF-8-SIG编码。')
for output_path in [*weighted_output_files.values(), *count_output_files.values()]:
    if output_path.read_bytes()[:3] != b'\xef\xbb\xbf':
        raise ValueError(f'{output_path.name}不是UTF-8-SIG编码。')

print('输出后核对全部通过。')
print(f'检查记录：{check_output_file}')
if backup_dir:
    print(f'旧结果备份：{backup_dir}')

输出后核对全部通过。
检查记录：C:\Users\Administrator\Desktop\W\Results\01_链路数据检查\链路数据检查.csv
旧结果备份：C:\Users\Administrator\Desktop\W\Archive\2026-10-17_214309_链路数据重跑前


In [6]:
display(Markdown('### 输出汇总'))
display(output_summary.style.format({'衰减权重合计': '{:,.6f}', '计数权重合计': '{:,.0f}'}).hide(axis='index'))

for year in analysis_years:
    display(Markdown(f'### {year}年衰减链路前10行'))
    display(weighted_outputs[year].head(10))

print('正式输出文件：')
for output_path in output_files:
    print(f'- {output_path.relative_to(project_root)}')

### 输出汇总

年份,81城内接续记录数,有向城市对数,衰减权重合计,计数权重合计
2021,96536,3281,"51,440.146837","96,536"
2022,257123,3653,"134,057.924479","257,123"
2023,441378,4348,"219,648.344314","441,378"


### 2021年衰减链路前10行

,年份,有向边编号,起点城市,终点城市,链路权重,权重口径,接续记录数,涉及农机数,一级边记录数,二级及以上边记录数,最小边级,最大边级,同城端点记录数,时间冲突记录数,是否同城边,来源文件
0,2021,三门峡市→三门峡市,三门峡市,三门峡市,3.87500,衰减权重求和,9,7,0,9,2,4,9,0,True,Data/03_有向边数据/2021年农机城市接续.csv
1,2021,三门峡市→临汾市,三门峡市,临汾市,1.50000,衰减权重求和,2,2,1,1,1,2,0,0,False,Data/03_有向边数据/2021年农机城市接续.csv
2,2021,三门峡市→保定市,三门峡市,保定市,1.28125,衰减权重求和,5,2,0,5,2,6,0,0,False,Data/03_有向边数据/2021年农机城市接续.csv
3,2021,三门峡市→咸阳市,三门峡市,咸阳市,1.25000,衰减权重求和,4,4,0,4,2,4,0,0,False,Data/03_有向边数据/2021年农机城市接续.csv
4,2021,三门峡市→宝鸡市,三门峡市,宝鸡市,4.50000,衰减权重求和,5,5,4,1,1,2,0,0,False,Data/03_有向边数据/2021年农机城市接续.csv
5,2021,三门峡市→平顶山市,三门峡市,平顶山市,1.00000,衰减权重求和,1,1,1,0,1,1,0,0,False,Data/03_有向边数据/2021年农机城市接续.csv
6,2021,三门峡市→晋城市,三门峡市,晋城市,1.00000,衰减权重求和,1,1,1,0,1,1,0,0,False,Data/03_有向边数据/2021年农机城市接续.csv
7,2021,三门峡市→洛阳市,三门峡市,洛阳市,7.75000,衰减权重求和,9,7,7,2,1,3,0,0,False,Data/03_有向边数据/2021年农机城市接续.csv
8,2021,三门峡市→渭南市,三门峡市,渭南市,5.50000,衰减权重求和,7,5,5,2,1,3,0,0,False,Data/03_有向边数据/2021年农机城市接续.csv
9,2021,三门峡市→濮阳市,三门峡市,濮阳市,1.25000,衰减权重求和,2,1,1,1,1,3,0,0,False,Data/03_有向边数据/2021年农机城市接续.csv


### 2022年衰减链路前10行

,年份,有向边编号,起点城市,终点城市,链路权重,权重口径,接续记录数,涉及农机数,一级边记录数,二级及以上边记录数,最小边级,最大边级,同城端点记录数,时间冲突记录数,是否同城边,来源文件
0,2022,三门峡市→三门峡市,三门峡市,三门峡市,9.50000,衰减权重求和,22,14,0,22,2,4,22,0,True,Data/03_有向边数据/2022年农机城市接续.csv
1,2022,三门峡市→临汾市,三门峡市,临汾市,13.37500,衰减权重求和,19,18,9,10,1,4,0,0,False,Data/03_有向边数据/2022年农机城市接续.csv
2,2022,三门峡市→咸阳市,三门峡市,咸阳市,6.53125,衰减权重求和,11,11,4,7,1,6,0,0,False,Data/03_有向边数据/2022年农机城市接续.csv
3,2022,三门峡市→安阳市,三门峡市,安阳市,1.00000,衰减权重求和,1,1,1,0,1,1,0,0,False,Data/03_有向边数据/2022年农机城市接续.csv
4,2022,三门峡市→宝鸡市,三门峡市,宝鸡市,10.62500,衰减权重求和,13,12,9,4,1,4,0,0,False,Data/03_有向边数据/2022年农机城市接续.csv
5,2022,三门峡市→晋城市,三门峡市,晋城市,5.50000,衰减权重求和,6,6,5,1,1,2,0,0,False,Data/03_有向边数据/2022年农机城市接续.csv
6,2022,三门峡市→洛阳市,三门峡市,洛阳市,18.00000,衰减权重求和,21,13,17,4,1,3,0,0,False,Data/03_有向边数据/2022年农机城市接续.csv
7,2022,三门峡市→济南市,三门峡市,济南市,0.50000,衰减权重求和,1,1,0,1,2,2,0,0,False,Data/03_有向边数据/2022年农机城市接续.csv
8,2022,三门峡市→渭南市,三门峡市,渭南市,3.00000,衰减权重求和,3,3,3,0,1,1,0,0,False,Data/03_有向边数据/2022年农机城市接续.csv
9,2022,三门峡市→濮阳市,三门峡市,濮阳市,1.00000,衰减权重求和,1,1,1,0,1,1,0,0,False,Data/03_有向边数据/2022年农机城市接续.csv


### 2023年衰减链路前10行

,年份,有向边编号,起点城市,终点城市,链路权重,权重口径,接续记录数,涉及农机数,一级边记录数,二级及以上边记录数,最小边级,最大边级,同城端点记录数,时间冲突记录数,是否同城边,来源文件
0,2023,三门峡市→三门峡市,三门峡市,三门峡市,44.611328,衰减权重求和,105,75,0,105,2,10,105,0,True,Data/03_有向边数据/2023年农机城市接续.csv
1,2023,三门峡市→临汾市,三门峡市,临汾市,129.992188,衰减权重求和,174,150,105,69,1,8,0,0,False,Data/03_有向边数据/2023年农机城市接续.csv
2,2023,三门峡市→保定市,三门峡市,保定市,5.000000,衰减权重求和,6,6,4,2,1,2,0,0,False,Data/03_有向边数据/2023年农机城市接续.csv
3,2023,三门峡市→信阳市,三门峡市,信阳市,1.000000,衰减权重求和,1,1,1,0,1,1,0,0,False,Data/03_有向边数据/2023年农机城市接续.csv
4,2023,三门峡市→周口市,三门峡市,周口市,3.000000,衰减权重求和,3,3,3,0,1,1,0,0,False,Data/03_有向边数据/2023年农机城市接续.csv
5,2023,三门峡市→咸阳市,三门峡市,咸阳市,47.832031,衰减权重求和,83,71,31,52,1,9,0,0,False,Data/03_有向边数据/2023年农机城市接续.csv
6,2023,三门峡市→唐山市,三门峡市,唐山市,2.000000,衰减权重求和,4,3,1,3,1,3,0,0,False,Data/03_有向边数据/2023年农机城市接续.csv
7,2023,三门峡市→商丘市,三门峡市,商丘市,1.000000,衰减权重求和,2,2,0,2,2,2,0,0,False,Data/03_有向边数据/2023年农机城市接续.csv
8,2023,三门峡市→天津市,三门峡市,天津市,3.820312,衰减权重求和,7,5,3,4,1,8,0,0,False,Data/03_有向边数据/2023年农机城市接续.csv
9,2023,三门峡市→威海市,三门峡市,威海市,1.250000,衰减权重求和,2,1,1,1,1,3,0,0,False,Data/03_有向边数据/2023年农机城市接续.csv


正式输出文件：
- Data\04_城市链路数据\2021年衰减链路.csv
- Data\04_城市链路数据\2022年衰减链路.csv
- Data\04_城市链路数据\2023年衰减链路.csv
- Data\04_城市链路数据\2021年计数链路.csv
- Data\04_城市链路数据\2022年计数链路.csv
- Data\04_城市链路数据\2023年计数链路.csv
- Results\01_链路数据检查\链路数据检查.csv
